# Module 2: The Solution - Document Intelligence

In **Module 1**, we broke our RAG pipeline using naive techniques on `metro-s36.pdf`:

| Failure | What We Saw in Module 1 |
|---------|-------------------------|
| **1. Table/Structure Loss** | Structured elements (legends, specifications) became jumbled text |
| **2. Figure Loss** | The zoning maps and station photos were completely invisible to our pipeline |
| **3. Context Loss** | Map coordinates, legend labels ("מגורים א׳", "תעסוקה") became orphaned without their visual context |

In this module, we will use **Azure AI Document Intelligence** (`prebuilt-layout` model) to fix these issues by extracting **structure** instead of just raw text.

## 🎯 Learning Objectives

By the end of this module, you will:
1. Understand how Document Intelligence preserves **table structure** (rows, columns, checkboxes)
2. See how it **detects figures** with bounding boxes (and how to crop them!)
3. Learn to **filter noise** (headers/footers) using paragraph roles
4. Appreciate why structure extraction is the foundation for quality RAG

In [ ]:
import os
import sys
from pathlib import Path

# Add the src directory to the path so we can import shared utilities
sys.path.append(str(Path("../../src").resolve()))

from utils import load_env
import pandas as pd
from IPython.display import Image, display, Markdown, HTML
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.identity import DefaultAzureCredential

# Load environment variables
env = load_env()

# Initialize Client with Entra ID (Keys are disabled in workshop resources)
endpoint = env["AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT"]

print("🔐 Initializing with DefaultAzureCredential (Entra ID)...")
credential = DefaultAzureCredential()
client = DocumentIntelligenceClient(endpoint=endpoint, credential=credential)

print("✅ Document Intelligence Client Ready.")

---

## Step 1: Analyze the Same Document from Module 1

We use the **same `metro-s36.pdf`** that failed in Module 1. This time, we'll use the `prebuilt-layout` model which doesn't just read text—it understands the **geometry** of the page.

**What `prebuilt-layout` extracts:**
- 📊 **Tables** with rows, columns, and header detection
- 🖼️ **Figures** with bounding box coordinates
- 📝 **Paragraphs** with semantic roles (header, footer, title, content)
- 📑 **Sections** with hierarchy information
- ✏️ **Markdown output** preserving structure

In [ ]:
# Use the SAME document from Module 1
DATA_DIR = Path("../../data/sample-pdfs")
PDF_PATH = DATA_DIR / "metro-s36.pdf"  # Same document that FAILED in Module 1!

print(f"📄 Analyzing {PDF_PATH.name} with prebuilt-layout...")
print("   This is the SAME document that broke our pipeline in Module 1!\n")
print("   ⏳ This may take 30-60 seconds...\n")

with open(PDF_PATH, "rb") as f:
    poller = client.begin_analyze_document(
        "prebuilt-layout", 
        f, 
        content_type="application/pdf"
    )
    result = poller.result()

print(f"✅ Analysis complete!")
print(f"   📑 Pages: {len(result.pages)}")
print(f"   📊 Tables found: {len(result.tables) if result.tables else 0}")
print(f"   🖼️ Figures found: {len(result.figures) if result.figures else 0}")
print(f"   📝 Paragraphs: {len(result.paragraphs) if result.paragraphs else 0}")

---

## 🔧 Solving Failure #1: Table Structure Preservation

### The Problem (from Module 1)

In Module 1, when we extracted tables as plain text, we lost the **relationship between cells**. Numbers became meaningless without their column context.

### The Solution: `result.tables`

Document Intelligence extracts tables with **full structure**—rows, columns, and even identifies which cells are headers!

Let's see what tables DI found in our document:

In [ ]:
# Let's examine ALL tables found in the document
print("📊 Tables Found by Document Intelligence:\n")

if result.tables:
    for i, table in enumerate(result.tables):
        page_num = table.bounding_regions[0].page_number if table.bounding_regions else "?"
        print(f"   Table {i+1}: {table.row_count} rows × {table.column_count} columns (Page {page_num})")
    
    print(f"\n💡 Found {len(result.tables)} table(s)!")
    print("   The TOC on Page 1-2 may be detected as multiple tables (one per page).")
else:
    print("   No tables found.")

### Reconstructing the TOC Table

Let's take the first table and reconstruct it as a proper DataFrame with column relationships preserved:

In [ ]:
# Let's look at ALL tables and understand what DI detected
print("📊 Examining All Detected Tables:\n")

if result.tables:
    for i, table in enumerate(result.tables):
        page_num = table.bounding_regions[0].page_number if table.bounding_regions else "?"
        
        print(f"{'='*60}")
        print(f"📋 Table {i+1} (Page {page_num}): {table.row_count} rows × {table.column_count} columns")
        print(f"{'='*60}")
        
        # Create a grid to hold the table data
        grid = [["" for _ in range(table.column_count)] for _ in range(table.row_count)]
        
        # Fill the grid with cell content
        for cell in table.cells:
            content = cell.content
            if cell.kind == "columnHeader":
                content = f"📌 {content}"
            grid[cell.row_index][cell.column_index] = content
        
        # Display as DataFrame
        df = pd.DataFrame(grid)
        display(df)
        print()
    
    print("💡 INSIGHT: DI detected structured elements like map legends!")
    print("   - ':selected:' and ':unselected:' indicate checkbox states")
    print("   - This preserves the meaning of visual indicators in the document")
    print("\n✅ SOLUTION: Table structure (rows/columns) is PRESERVED!")
    print("   Even checkbox states from map legends are captured properly.")
else:
    print("No tables found in this document.")

---

## 🔧 Solving Failure #2: Missing Figures (Maps & Diagrams)

### The Problem (from Module 1)

In Module 1, when we asked "What is the zoning east of Station 36?", we got legend labels like `מגורים א׳, תעסוקה, מסחר` floating in the text—but **the actual MAP was invisible**.

The answer requires SEEING the colored regions on the map, not just reading disconnected labels.

### The Solution: `result.figures`

Document Intelligence detects figures and provides their **bounding box coordinates**. While it doesn't describe the image content (that's Module 3 with GPT-4o Vision!), it tells us:
- **WHERE** the figures are located
- **What captions** they have (if any)
- **Coordinates** we can use to crop and send to a Vision model

In [ ]:
# Let's examine ALL figures found in the document
print("🖼️ Figures Found by Document Intelligence:\n")

if result.figures:
    # Group figures by page
    figures_by_page = {}
    for fig in result.figures:
        page = fig.bounding_regions[0].page_number if fig.bounding_regions else 0
        if page not in figures_by_page:
            figures_by_page[page] = []
        figures_by_page[page].append(fig)
    
    # Display summary
    for page, figs in sorted(figures_by_page.items()):
        print(f"   📄 Page {page}: {len(figs)} figure(s)")
        for i, fig in enumerate(figs):
            caption_text = fig.caption.content if fig.caption else "(no caption)"
            print(f"      └─ Figure: {caption_text[:50]}...")
    
    print(f"\n✅ Total: {len(result.figures)} figures detected!")
    print("   These could be maps, diagrams, charts, or images.")
    print("   In Module 3, we'll use GPT-4o Vision to DESCRIBE these figures!")
else:
    print("   No figures detected by the Layout model.")
    print("   Note: Some documents have embedded images that require different extraction.")

In [ ]:
# Let's look at figures on Page 1 specifically - the station overview page
# This page has: 1 map + 3 street photos = 4 potential figures to crop!

target_page = 1
page_figures = [
    f for f in result.figures 
    if f.bounding_regions and f.bounding_regions[0].page_number == target_page
]

print(f"🖼️ Figures on Page {target_page} (Station Overview):\n")
print(f"   Found {len(page_figures)} figure(s)\n")

for i, fig in enumerate(page_figures):
    polygon = fig.bounding_regions[0].polygon if fig.bounding_regions else []
    caption = fig.caption.content if fig.caption else "(no caption)"
    
    print(f"   📷 Figure {i+1}:")
    print(f"      Caption: {caption[:60]}...")
    print(f"      Bounding Box: {len(polygon)} coordinates (4 corners)")
    if polygon:
        # Polygon is [x1,y1, x2,y2, x3,y3, x4,y4] - 8 values for 4 corners
        print(f"      Top-Left:     ({polygon[0]:.2f}, {polygon[1]:.2f})")
        print(f"      Top-Right:    ({polygon[2]:.2f}, {polygon[3]:.2f})")
        print(f"      Bottom-Right: ({polygon[4]:.2f}, {polygon[5]:.2f})")
        print(f"      Bottom-Left:  ({polygon[6]:.2f}, {polygon[7]:.2f})")
    print()

### 📐 How Image Cropping Works with Bounding Boxes

Looking at **Page 1** of `metro-s36.pdf`, we can see 4 visual elements:

```
┌────────────────────────────────────────────────────────────────┐
│                                                                │
│   ┌──────────────────────┐      ┌──────────────────────────┐  │
│   │                      │      │                          │  │
│   │   🗺️ MAP             │      │   Station Info (text)    │  │
│   │   (800m radius)      │      │   - מיקום התחנה          │  │
│   │                      │      │   - סוג תחנה             │  │
│   │   Figure #1          │      │   - קיבולת נוסעים        │  │
│   │                      │      │                          │  │
│   └──────────────────────┘      └──────────────────────────┘  │
│                                                                │
│   ┌────────────┐  ┌────────────┐  ┌────────────┐              │
│   │   📷 3     │  │   📷 2     │  │   📷 1     │              │
│   │  North     │  │  South     │  │  East      │              │
│   │  View      │  │  View      │  │  View      │              │
│   │ Figure #2  │  │ Figure #3  │  │ Figure #4  │              │
│   └────────────┘  └────────────┘  └────────────┘              │
│                                                                │
│   1. מבט לכיוון מזרח   2. מבט לכיוון דרום   3. מבט לכיוון צפון │
└────────────────────────────────────────────────────────────────┘
```

**Document Intelligence gives us the polygon coordinates for each figure!**

We can use these coordinates to:
1. **Render the PDF page** as an image (using `pdf2image` or similar)
2. **Crop each figure** using the bounding box coordinates
3. **Send to GPT-4o Vision** for AI description
4. **Index the description** alongside the text content

This is exactly what we do in **Module 7's production pipeline**!

In [ ]:
# Let's demonstrate how to crop a figure using the bounding box
# This is a PREVIEW of what we do in Module 7's pipeline

def demo_crop_figure(fig, page_width=8.5, page_height=11):
    """
    Demonstrate how to calculate crop coordinates from DI polygon.
    
    DI provides coordinates in INCHES from top-left origin.
    To crop an image, we need to convert to pixel coordinates.
    """
    if not fig.bounding_regions:
        return None
    
    polygon = fig.bounding_regions[0].polygon
    page_num = fig.bounding_regions[0].page_number
    
    # Extract bounding box from polygon (min/max of all corners)
    x_coords = [polygon[i] for i in range(0, len(polygon), 2)]
    y_coords = [polygon[i] for i in range(1, len(polygon), 2)]
    
    left = min(x_coords)
    right = max(x_coords)
    top = min(y_coords)
    bottom = max(y_coords)
    
    return {
        "page": page_num,
        "left_inches": left,
        "top_inches": top,
        "right_inches": right,
        "bottom_inches": bottom,
        "width_inches": right - left,
        "height_inches": bottom - top
    }

# Show crop info for each figure on Page 1
print("✂️ Crop Coordinates for Page 1 Figures:\n")
print("   These coordinates can be used to extract images from the PDF!\n")

for i, fig in enumerate(page_figures):
    crop_info = demo_crop_figure(fig)
    if crop_info:
        print(f"   📷 Figure {i+1}:")
        print(f"      Position: ({crop_info['left_inches']:.2f}\", {crop_info['top_inches']:.2f}\") from top-left")
        print(f"      Size: {crop_info['width_inches']:.2f}\" × {crop_info['height_inches']:.2f}\"")
        print()

print("💡 In Module 7, we use these coordinates to:")
print("   1. Render PDF page at 300 DPI → PIL Image")
print("   2. Convert inches to pixels: pixels = inches × 300")
print("   3. Crop: image.crop((left_px, top_px, right_px, bottom_px))")
print("   4. Send cropped image to GPT-4o Vision for description")

In [ ]:
# BONUS: Full working example of image cropping (requires pdf2image + poppler)
# This is the ACTUAL code pattern used in Module 7's pipeline

cropping_code = """
# ═══════════════════════════════════════════════════════════════════
# PRODUCTION CODE: How Module 7 crops figures from PDFs
# ═══════════════════════════════════════════════════════════════════

from pdf2image import convert_from_path
from PIL import Image

def crop_figures_from_pdf(pdf_path, figures, dpi=300):
    '''
    Crop all detected figures from a PDF using DI bounding boxes.
    
    Args:
        pdf_path: Path to the PDF file
        figures: List of figures from DI result.figures
        dpi: Resolution for rendering (300 = print quality)
    
    Returns:
        List of cropped PIL Images
    '''
    # Render all PDF pages as images
    pages = convert_from_path(pdf_path, dpi=dpi)
    
    cropped_images = []
    
    for fig in figures:
        if not fig.bounding_regions:
            continue
            
        page_num = fig.bounding_regions[0].page_number
        polygon = fig.bounding_regions[0].polygon
        
        # Get the page image (0-indexed)
        page_image = pages[page_num - 1]
        
        # Convert inches to pixels
        x_coords = [polygon[i] * dpi for i in range(0, len(polygon), 2)]
        y_coords = [polygon[i] * dpi for i in range(1, len(polygon), 2)]
        
        left = int(min(x_coords))
        top = int(min(y_coords))
        right = int(max(x_coords))
        bottom = int(max(y_coords))
        
        # Crop the figure
        cropped = page_image.crop((left, top, right, bottom))
        cropped_images.append({
            'image': cropped,
            'page': page_num,
            'caption': fig.caption.content if fig.caption else None
        })
    
    return cropped_images

# Usage:
# cropped = crop_figures_from_pdf('metro-s36.pdf', result.figures)
# cropped[0]['image'].save('figure_1.png')  # Save the map
# cropped[1]['image'].save('figure_2.png')  # Save street photo 1
"""

print("📋 Production Code for Image Cropping:")
print("=" * 65)
print(cropping_code)
print("=" * 65)
print("\n⚠️  Note: This requires 'pdf2image' and 'poppler' to be installed.")
print("   In Module 7, we handle this in the backend pipeline.")

---

## 🔧 Solving Failure #3: Context Loss (Headers/Footers as Noise)

### The Problem (from Module 1)

In Module 1, chunks included repeated headers and footers mixed with actual content:
- Architect names from page footers
- Project identifiers and dates
- Page numbers floating randomly

This noise confused the LLM and diluted the relevance of our chunks.

### The Solution: Paragraph Roles

Document Intelligence identifies **semantic roles** for paragraphs:
- `pageHeader` / `pageFooter` - Navigation/metadata we should EXCLUDE
- `title` / `sectionHeading` - Structural markers we should KEEP as context
- `content` - The actual information to index

In [ ]:
# Let's examine paragraph roles across the document
print("📝 Analyzing Paragraph Roles:\n")

if result.paragraphs:
    # Count paragraphs by role
    role_counts = {}
    for p in result.paragraphs:
        # Handle both string and enum-style role values
        role = str(p.role) if p.role else "content"
        # Simplify role names for display
        role_display = role.replace("ParagraphRole.", "").lower()
        role_counts[role_display] = role_counts.get(role_display, 0) + 1
    
    print("   Role Distribution:")
    for role, count in sorted(role_counts.items()):
        emoji = "🚫" if "footer" in role or "header" in role else "✅"
        print(f"      {emoji} {role}: {count} paragraphs")
    
    print("\n   💡 Strategy: EXCLUDE page_footer/page_header from indexing!")
else:
    print("   No paragraphs found.")

In [ ]:
# Let's look at specific examples of noise vs content from Page 1
target_page = 1
print(f"📄 Examining Paragraphs on Page {target_page}:\n")

page_paragraphs = [
    p for p in result.paragraphs 
    if p.bounding_regions and p.bounding_regions[0].page_number == target_page
]

print(f"Found {len(page_paragraphs)} paragraphs on Page {target_page}:\n")

for p in page_paragraphs[:10]:  # Show first 10
    # Handle both string and enum-style role values
    role = str(p.role) if p.role else "content"
    role_display = role.replace("ParagraphRole.", "").lower()
    content_preview = p.content[:60].replace('\n', ' ')
    
    if "footer" in role_display or "header" in role_display:
        print(f"   🚫 [{role_display.upper()}] '{content_preview}...' → SKIP!")
    else:
        print(f"   ✅ [{role_display.upper()}] '{content_preview}...' → INDEX")

print("\n✅ SOLUTION: Filter by role to exclude navigation noise from your index!")

---

## 📊 Summary: What Document Intelligence Gives Us

We've addressed all three failures from Module 1's naive approach:

| Failure | Module 1 (Naive) | Module 2 (Document Intelligence) |
|---------|------------------|----------------------------------|
| **Tables** | TOC became jumbled text—page numbers lost | ✅ `result.tables` preserves rows, columns, headers |
| **Figures** | Maps completely invisible | ✅ `result.figures` detects location + bounding boxes |
| **Noise** | Headers/footers mixed with content | ✅ `paragraph.role` lets us filter noise |

### 🎯 Key Takeaways

1. **Document Intelligence extracts STRUCTURE, not just text**
   - Tables retain their row/column relationships
   - Figures are detected with coordinates
   - Paragraphs have semantic roles

2. **This is the FOUNDATION for quality RAG**
   - You can't chunk intelligently without knowing what's a table vs. paragraph
   - You can't answer figure questions without detecting figures exist
   - You can't get clean chunks without filtering noise

3. **But we're not done yet!**
   - DI detects figures, but doesn't DESCRIBE them (that needs Vision AI)
   - DI gives us structure, but we still need smart chunking strategies
   - That's what we'll tackle in Module 3 and Module 4!

---

## ➡️ What's Next?

| Module | What You'll Learn |
|--------|-------------------|
| **Module 3** | Content Understanding - AI descriptions for figures, semantic extraction |
| **Module 4** | Smart Chunking - Using structure to create meaningful chunks |
| **Module 5** | Azure AI Search - Hybrid search with the structured content |

**Next**: [Module 3 - Content Understanding](../module-3-content-understanding/README.md)